### 실습 1. Chapter 01 전처리 데이터 불러오기

이번 Chapter에서도 Chapter 01에서 만든 다음 파일을 사용합니다.

In [3]:
import pandas as pd

DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())

df_books[["상품명"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류
5,초역 부처의 말
6,청춘의 독서(특별증보판)
7,어른의 행복은 조용하다
8,채식주의자
9,단 한 번의 삶(강물에디션 활판인쇄 한정판)


📋 실행 결과 체크리스트 검토

파일이 정상적으로 열렸는가?

예: pd.read_csv를 통해 데이터가 오류 없이 정상적으로 로드되었으며, 하단에 데이터 크기: (199, 8)로 정상 출력된 것을 확인할 수 있습니다.

상품명 컬럼이 존재하는가?

예: 출력된 컬럼 목록('순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야')에 '상품명'이 포함되어 있으며, 상품명 결측치: 0으로 데이터 이상 없이 존재합니다.

한글이 정상적으로 보이는가?

예: encoding="utf-8-sig" 옵션이 적용되어 '소년이 온다', '모순', '결국 국민이 합니다' 등 도서 제목의 한글이 깨짐 없이 선명하게 잘 출력되고 있습니다.

Chapter 01 결과와 행 개수가 크게 달라지지 않았는가?

예: 데이터 총 행 개수가 199행으로 정상적으로 집계되었으며, 이전 Chapter 01의 전처리 결과와 비교했을 때 행 개수 손실이나 큰 변동 없이 유지되고 있음을 확인할 수 있습니다.

종합 의견: 모든 검토 항목이 정상적으로 충족되었으므로, 다음 단계를 진행하셔도 좋습니다!

### 실습 2. 분석할 제목 문자열 준비하기
 
Vectorizer에 전달할 제목을 문자열 형태로 준비합니다.

In [4]:
titles = (

    df_books["상품명"]

    .fillna("")

    .astype(str)

    .str.strip()

)

titles = titles[titles != ""].reset_index(drop=True)

print("사용할 제목 수:", len(titles))

titles.head(10)

사용할 제목 수: 199


0                      소년이 온다
1                          모순
2                  결국 국민이 합니다
3                         혼모노
4                          급류
5                    초역 부처의 말
6               청춘의 독서(특별증보판)
7                어른의 행복은 조용하다
8                       채식주의자
9    단 한 번의 삶(강물에디션 활판인쇄 한정판)
Name: 상품명, dtype: str

### 실습 3. 머신러닝은 왜 텍스트를 숫자로 바꿀까?

머신러닝 알고리즘은 일반적으로 숫자를 입력받아 계산합니다.
 
예를 들어 다음과 같은 숫자는 계산할 수 있습니다.

도서 제목

        ↓

사용된 단어 확인

        ↓

단어마다 열(column) 생성

        ↓

등장 여부 또는 등장 횟수를 숫자로 기록

        ↓

숫자 벡터

### 실습 4. 아주 작은 예제로 먼저 이해하기
 
실제 1,000권 정도의 도서 제목을 바로 사용하면 행과 열이 많아져 원리를 보기 어렵습니다.
 
먼저 세 문장만 사용합니다.

In [5]:
# 실습 4. 아주 작은 예제로 먼저 이해하기

# 1. 세 개의 작은 문장 정의하기
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문",
]

# 2. 고유 단어 사전(단어 순서) 정의
vocabulary = ["데이터", "머신러닝", "분석", "입문", "파이썬"]

print("샘플 문서:", sample_docs)
print("단어 순서(사전):", vocabulary)

# 3. 첫 번째 문장("파이썬 데이터 분석")을 단어 출현 횟수 벡터로 표현하기
# [데이터, 머신러닝, 분석, 입문, 파이썬] 순서에 따른 카운트
first_doc_vector = [1, 0, 1, 0, 1]

print("첫 번째 문장 벡터 표현:", first_doc_vector)

샘플 문서: ['파이썬 데이터 분석', '파이썬 머신러닝', '데이터 분석 입문']
단어 순서(사전): ['데이터', '머신러닝', '분석', '입문', '파이썬']
첫 번째 문장 벡터 표현: [1, 0, 1, 0, 1]


### 실습 5. Bag of Words 이해하기
 
앞의 방식은 단어를 하나의 주머니에 넣고 몇 번 등장했는지 세는 것과 비슷합니다.
 
이 개념을 Bag of Words, 줄여서 BoW라고 부릅니다.

### 실습 6. CountVectorizer 설치 확인하기
 
CountVectorizer는 scikit-learn에 포함되어 있습니다.
 
설치가 필요한 경우 다음과 같이 실행합니다.

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer()

X_count_sample = count_vectorizer.fit_transform(sample_docs)

In [7]:
feature_names = count_vectorizer.get_feature_names_out()

print(feature_names)

['데이터' '머신러닝' '분석' '입문' '파이썬']


In [8]:

X_count_sample.toarray()

array([[1, 0, 1, 0, 1],
       [0, 1, 0, 0, 1],
       [1, 0, 1, 1, 0]])

In [9]:
sample_count_df = pd.DataFrame(

    X_count_sample.toarray(),

    columns=feature_names,

    index=sample_docs,

)

sample_count_df

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


In [10]:
repeat_docs = [

    "파이썬 파이썬 데이터",

    "데이터 분석",

]

repeat_vectorizer = CountVectorizer()

X_repeat = repeat_vectorizer.fit_transform(repeat_docs)

repeat_df = pd.DataFrame(

    X_repeat.toarray(),

    columns=repeat_vectorizer.get_feature_names_out(),

    index=repeat_docs,

)

repeat_df

,데이터,분석,파이썬
파이썬 파이썬 데이터,1,0,2
데이터 분석,1,1,0


In [11]:
test_docs = [

    "AI 데이터 분석 R 파이썬",

]

test_vectorizer = CountVectorizer()

X_test = test_vectorizer.fit_transform(test_docs)

print(test_vectorizer.get_feature_names_out())

['ai' '데이터' '분석' '파이썬']


In [12]:
count_vectorizer = CountVectorizer()

X_count = count_vectorizer.fit_transform(titles)

count_terms = count_vectorizer.get_feature_names_out()

In [13]:

print("문서 수:", X_count.shape[0])

print("단어 수:", X_count.shape[1])

print("행렬 크기:", X_count.shape)

문서 수: 199
단어 수: 536
행렬 크기: (199, 536)


In [14]:

list(count_vectorizer.vocabulary_.items())[:20]

[('소년이', 278),
 ('온다', 355),
 ('모순', 194),
 ('결국', 67),
 ('국민이', 84),
 ('합니다', 511),
 ('혼모노', 524),
 ('급류', 92),
 ('초역', 460),
 ('부처의', 236),
 ('청춘의', 458),
 ('독서', 150),
 ('특별증보판', 485),
 ('어른의', 332),
 ('행복은', 517),
 ('조용하다', 432),
 ('채식주의자', 454),
 ('번의', 220),
 ('강물에디션', 57),
 ('활판인쇄', 528)]

In [15]:

type(X_count)

scipy.sparse._csr.csr_matrix

In [16]:
X_count_sample.toarray()

array([[1, 0, 1, 0, 1],
       [0, 1, 0, 0, 1],
       [1, 0, 1, 1, 0]])

In [17]:
print(titles.iloc[0])

소년이 온다


In [18]:

first_row = X_count.getrow(0)

indices = first_row.indices

values = first_row.data

first_title_terms = [

    (count_terms[index], value)

    for index, value in zip(indices, values)

]

first_title_terms

[('소년이', np.int64(1)), ('온다', np.int64(1))]

In [19]:

import numpy as np

count_sums = np.asarray(X_count.sum(axis=0)).ravel()

count_summary = pd.DataFrame({

    "단어": count_terms,

    "전체등장횟수": count_sums,

})

count_summary = count_summary.sort_values(

    "전체등장횟수",

    ascending=False,

).reset_index(drop=True)

count_summary.head(30)

,단어,전체등장횟수
0,에디션,12
1,리커버,8
2,기념,8
3,위한,7
4,해커스,7
5,토익,6
6,나는,5
7,내가,5
8,스페셜,5
9,싶은,4


In [20]:

count_top30 = count_summary.head(30)

count_top30.to_csv(

    "chapter03_count_top_terms.csv",

    index=False,

    encoding="utf-8-sig",

)

### 실습 23. 작은 예제로 TfidfVectorizer 적용하기

In [21]:

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_sample_vectorizer = TfidfVectorizer()

X_tfidf_sample = tfidf_sample_vectorizer.fit_transform(sample_docs)

tfidf_sample_terms = tfidf_sample_vectorizer.get_feature_names_out()

In [22]:

tfidf_sample_df = pd.DataFrame(

    X_tfidf_sample.toarray(),

    columns=tfidf_sample_terms,

    index=sample_docs,

)

tfidf_sample_df.round(3)

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,0.577,0.000,0.577,0.000,0.577
파이썬 머신러닝,0.000,0.796,0.000,0.000,0.605
데이터 분석 입문,0.518,0.000,0.518,0.681,0.000


In [23]:
print(tfidf_sample_vectorizer.get_feature_names_out())

['데이터' '머신러닝' '분석' '입문' '파이썬']


In [24]:

idf_df = pd.DataFrame({

    "단어": tfidf_sample_vectorizer.get_feature_names_out(),

    "IDF": tfidf_sample_vectorizer.idf_,

})

idf_df.sort_values("IDF", ascending=False)

,단어,IDF
1,머신러닝,1.693147
3,입문,1.693147
0,데이터,1.287682
2,분석,1.287682
4,파이썬,1.287682


In [25]:

tfidf_vectorizer = TfidfVectorizer()

X_tfidf = tfidf_vectorizer.fit_transform(titles)

tfidf_terms = tfidf_vectorizer.get_feature_names_out()

In [26]:
print("TF-IDF 행렬 크기:", X_tfidf.shape)

print("문서 수:", X_tfidf.shape[0])

print("단어 수:", X_tfidf.shape[1])

TF-IDF 행렬 크기: (199, 536)
문서 수: 199
단어 수: 536


In [27]:
print("Count shape:", X_count.shape)

print("TF-IDF shape:", X_tfidf.shape)

Count shape: (199, 536)
TF-IDF shape: (199, 536)


### 실습 27. 첫 번째 도서의 TF-IDF 주요 단어 확인하기
 
첫 번째 제목을 다시 확인합니다.

In [28]:
print("도서 제목:", titles.iloc[0])

도서 제목: 소년이 온다


In [29]:
first_tfidf_row = X_tfidf.getrow(0)

first_tfidf_df = pd.DataFrame({

    "단어": tfidf_terms[first_tfidf_row.indices],

    "TF-IDF": first_tfidf_row.data,

})

first_tfidf_df = first_tfidf_df.sort_values(

    "TF-IDF",

    ascending=False,

).reset_index(drop=True)

first_tfidf_df

,단어,TF-IDF
0,소년이,0.707107
1,온다,0.707107


### 실습 28. 여러 도서의 주요 TF-IDF 단어 확인 함수 만들기
 
반복해서 확인하기 위해 작은 함수를 만들 수 있습니다.

In [31]:
def show_top_tfidf_terms(doc_index, top_n=5):

    row = X_tfidf.getrow(doc_index)

    result = pd.DataFrame({

        "단어": tfidf_terms[row.indices],

        "TF-IDF": row.data,

    })

    result = result.sort_values(

        "TF-IDF",

        ascending=False,

    ).head(top_n)

    print("도서 제목:", titles.iloc[doc_index])

    return result.reset_index(drop=True)

In [33]:
show_top_tfidf_terms(100, top_n=5)

도서 제목: 말투만 바꿨을 뿐인데


,단어,TF-IDF
0,말투만,0.57735
1,바꿨을,0.57735
2,뿐인데,0.57735


In [34]:
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

tfidf_summary = pd.DataFrame({

    "단어": tfidf_terms,

    "평균_TFIDF": mean_tfidf,

})

tfidf_summary = tfidf_summary.sort_values(

    "평균_TFIDF",

    ascending=False,

).reset_index(drop=True)

tfidf_summary.head(30)

,단어,평균_TFIDF
0,에디션,0.019330
1,리커버,0.013861
2,기념,0.012711
3,위한,0.012087
4,해커스,0.012072
5,토익,0.010950
6,스페셜,0.009901
7,나는,0.009786
8,내가,0.009454
9,대하여,0.008822


### 실습 30. TF-IDF 상위 단어 저장하기

In [ ]:
tfidf_top30 = tfidf_summary.head(30)

tfidf_top30.to_csv(

    "chapter03_tfidf_top_terms.csv",

    index=False,

    encoding="utf-8-sig",

)

### 실습 31. Count 상위 단어와 TF-IDF 상위 단어 비교하기
 
두 결과를 나란히 비교합니다.

In [36]:
comparison = pd.DataFrame({

    "Count_상위단어": count_top30["단어"].reset_index(drop=True),

    "Count_전체등장횟수": count_top30["전체등장횟수"].reset_index(drop=True),

    "TFIDF_상위단어": tfidf_top30["단어"].reset_index(drop=True),

    "평균_TFIDF": tfidf_top30["평균_TFIDF"].reset_index(drop=True),

})

comparison.head(20)

,Count_상위단어,Count_전체등장횟수,TFIDF_상위단어,평균_TFIDF
0,에디션,12,에디션,0.019330
1,리커버,8,리커버,0.013861
2,기념,8,기념,0.012711
3,위한,7,위한,0.012087
4,해커스,7,해커스,0.012072
5,토익,6,토익,0.010950
6,나는,5,스페셜,0.009901
7,내가,5,나는,0.009786
8,스페셜,5,내가,0.009454
9,싶은,4,대하여,0.008822


### 실습 47. 일부 도서 결과를 직접 검증하기
 
무작위로 몇 개 제목을 선택해 TF-IDF 주요 단어를 확인해 봅니다.

In [ ]:
sample_indices = [0, 10, 20]

for index in sample_indices:

    if index < len(titles):

        print("=" * 60)

        print("도서 제목:", titles.iloc[index])

        display(show_top_tfidf_terms(index, top_n=5))

도서 제목: 소년이 온다
도서 제목: 소년이 온다


,단어,TF-IDF
0,소년이,0.707107
1,온다,0.707107


도서 제목: 작별하지 않는다
도서 제목: 작별하지 않는다


,단어,TF-IDF
0,작별하지,0.752078
1,않는다,0.659074


도서 제목: 쇼펜하우어 인생수업(30만 부 기념 개정증보판)
도서 제목: 쇼펜하우어 인생수업(30만 부 기념 개정증보판)


,단어,TF-IDF
0,개정증보판,0.498481
1,쇼펜하우어,0.462422
2,인생수업,0.462422
3,30만,0.436838
4,기념,0.364720
